In [1]:
print("Day - 07 Window functions (LEAD, LAG, ROW_NUMBER for sequences)")

Day - 07 Window functions (LEAD, LAG, ROW_NUMBER for sequences)


SQL Window Functions for Feature Engineering
(LEAD, LAG, ROW_NUMBER)

Think of window functions as:

“Do calculations across related rows without collapsing them like GROUP BY does.”

This is exactly what we need for time series, sequences, and behavior modeling.

1️⃣ What is a Window Function (Big Picture)

Normal SQL:

GROUP BY → reduces rows ❌

Window functions:

Keep all rows

Add new features based on neighboring rows ✅

Syntax pattern (important):

FUNCTION(column) OVER (
    PARTITION BY ...
    ORDER BY ...
)


PARTITION BY → group (like per user)

ORDER BY → sequence (time / order)

In [2]:
import sqlite3
import pandas as pd
import numpy as np

In [3]:
conn = sqlite3.connect(":memory:")

In [4]:
np.random.seed(42)

data = []

for user_id in range(1, 4):  # 3 users
    dates = pd.date_range("2026-01-01", periods=5, freq="D")
    amounts = np.random.randint(50, 300, size=5)
    
    for d, a in zip(dates, amounts):
        data.append([user_id, d.strftime("%Y-%m-%d"), a])

df = pd.DataFrame(data, columns=["user_id", "date", "amount"])
df


,user_id,date,amount
0,1,2026-01-01,152
1,1,2026-01-02,229
2,1,2026-01-03,142
3,1,2026-01-04,64
4,1,2026-01-05,156
5,2,2026-01-01,121
6,2,2026-01-02,238
7,2,2026-01-03,70
8,2,2026-01-04,152
9,2,2026-01-05,171


In [5]:
df.to_sql("transactions", conn, index=False, if_exists="replace")

15

In [6]:
query = """
SELECT
    user_id,
    date,
    amount,
    LAG(amount) OVER (
        PARTITION BY user_id
        ORDER BY date
    ) AS prev_amount
FROM transactions;
"""

pd.read_sql(query, conn)


,user_id,date,amount,prev_amount
0,1,2026-01-01,152,NaN
1,1,2026-01-02,229,152.0
2,1,2026-01-03,142,229.0
3,1,2026-01-04,64,142.0
4,1,2026-01-05,156,64.0
5,2,2026-01-01,121,NaN
6,2,2026-01-02,238,121.0
7,2,2026-01-03,70,238.0
8,2,2026-01-04,152,70.0
9,2,2026-01-05,171,152.0


In [7]:
query = """
SELECT
    user_id,
    date,
    amount,
    LEAD(amount) OVER (
        PARTITION BY user_id
        ORDER BY date
    ) AS next_amount
FROM transactions;
"""

pd.read_sql(query, conn)


,user_id,date,amount,next_amount
0,1,2026-01-01,152,229.0
1,1,2026-01-02,229,142.0
2,1,2026-01-03,142,64.0
3,1,2026-01-04,64,156.0
4,1,2026-01-05,156,NaN
5,2,2026-01-01,121,238.0
6,2,2026-01-02,238,70.0
7,2,2026-01-03,70,152.0
8,2,2026-01-04,152,171.0
9,2,2026-01-05,171,NaN


In [8]:
query = """
SELECT
    user_id,
    date,
    amount,
    ROW_NUMBER() OVER (
        PARTITION BY user_id
        ORDER BY date
    ) AS event_number
FROM transactions;
"""

pd.read_sql(query, conn)


,user_id,date,amount,event_number
0,1,2026-01-01,152,1
1,1,2026-01-02,229,2
2,1,2026-01-03,142,3
3,1,2026-01-04,64,4
4,1,2026-01-05,156,5
5,2,2026-01-01,121,1
6,2,2026-01-02,238,2
7,2,2026-01-03,70,3
8,2,2026-01-04,152,4
9,2,2026-01-05,171,5


In [9]:
query = """
SELECT
    user_id,
    date,
    amount,
    amount - LAG(amount) OVER (
        PARTITION BY user_id
        ORDER BY date
    ) AS daily_change
FROM transactions;
"""

pd.read_sql(query, conn)


,user_id,date,amount,daily_change
0,1,2026-01-01,152,NaN
1,1,2026-01-02,229,77.0
2,1,2026-01-03,142,-87.0
3,1,2026-01-04,64,-78.0
4,1,2026-01-05,156,92.0
5,2,2026-01-01,121,NaN
6,2,2026-01-02,238,117.0
7,2,2026-01-03,70,-168.0
8,2,2026-01-04,152,82.0
9,2,2026-01-05,171,19.0


In [10]:
query = """
SELECT *,
CASE 
    WHEN ROW_NUMBER() OVER (
        PARTITION BY user_id
        ORDER BY date
    ) = 1 THEN 1
    ELSE 0
END AS is_first_event
FROM transactions;
"""

pd.read_sql(query, conn)


,user_id,date,amount,is_first_event
0,1,2026-01-01,152,1
1,1,2026-01-02,229,0
2,1,2026-01-03,142,0
3,1,2026-01-04,64,0
4,1,2026-01-05,156,0
5,2,2026-01-01,121,1
6,2,2026-01-02,238,0
7,2,2026-01-03,70,0
8,2,2026-01-04,152,0
9,2,2026-01-05,171,0


SQL CTEs for Complex Feature Derivation

(WITH clause explained properly)

1️⃣ What is a CTE? (Plain English)

CTE = Common Table Expression

In simple words:

A CTE is a temporary named table created inside a query.

It exists only for that query.

Think of it as:

A step-by-step pipeline

Like assigning variables in Python

Or intermediate DataFrames in pandas

2️⃣ Why CTEs are CRITICAL for Feature Engineering

Without CTEs:

SQL becomes long, unreadable, and confusing

With CTEs:

Each feature is built step-by-step

Easy to debug

Easy to explain in interviews

In [16]:
query = """
WITH lag_features AS (
    SELECT
        user_id,
        date,
        amount,
        LAG(amount) OVER (
            PARTITION BY user_id
            ORDER BY date
        ) AS prev_amount
    FROM transactions
)

SELECT *
FROM lag_features;
"""

pd.read_sql(query, conn)

,user_id,date,amount,prev_amount
0,1,2026-01-01,152,NaN
1,1,2026-01-02,229,152.0
2,1,2026-01-03,142,229.0
3,1,2026-01-04,64,142.0
4,1,2026-01-05,156,64.0
5,2,2026-01-01,121,NaN
6,2,2026-01-02,238,121.0
7,2,2026-01-03,70,238.0
8,2,2026-01-04,152,70.0
9,2,2026-01-05,171,152.0


In [17]:
# What’s happening:

# lag_features → temporary table
# It already contains prev_amount
# Final SELECT just reads it
# 📌 This is like:
# df_lag = df.assign(prev_amount=...)

5️⃣ Why NOT do everything in one SELECT?

❌ BAD SQL (hard to read):

SELECT
    user_id,
    date,
    amount,
    amount - LAG(amount) OVER (
        PARTITION BY user_id ORDER BY date
    ) AS daily_change,
    CASE 
        WHEN LAG(amount) OVER (
            PARTITION BY user_id ORDER BY date
        ) IS NULL THEN 1
        ELSE 0
    END AS is_first
FROM transactions;


👎 Repeating window functions
👎 Hard to debug
👎 Interviewers hate this

6️⃣ GOOD SQL with CTE (Industry Style)
WITH lag_features AS (
    SELECT
        user_id,
        date,
        amount,
        LAG(amount) OVER (
            PARTITION BY user_id
            ORDER BY date
        ) AS prev_amount
    FROM transactions
)

SELECT
    user_id,
    date,
    amount,
    amount - prev_amount AS daily_change,
    CASE
        WHEN prev_amount IS NULL THEN 1
        ELSE 0
    END AS is_first_event
FROM lag_features;


🔥 Clean
🔥 Readable
🔥 Reusable

7️⃣ Multiple CTEs (REAL FEATURE PIPELINE)
Goal:

1️⃣ Create lag
2️⃣ Create sequence number
3️⃣ Create final features

WITH base AS (
    SELECT
        user_id,
        date,
        amount
    FROM transactions
),

lagged AS (
    SELECT
        *,
        LAG(amount) OVER (
            PARTITION BY user_id
            ORDER BY date
        ) AS prev_amount
    FROM base
),

ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY date
        ) AS event_number
    FROM lagged
)

SELECT
    user_id,
    date,
    amount,
    prev_amount,
    amount - prev_amount AS daily_change,
    CASE
        WHEN event_number = 1 THEN 1
        ELSE 0
    END AS is_first_event
FROM ranked;


🧠 This is feature engineering in SQL, step-by-step.

8️⃣ CTEs vs Subqueries (Why CTEs Win)
Feature     	    CTE         Subquery
Readability	        ⭐⭐⭐⭐⭐	⭐⭐
Reuse	            Yes         No
Debugging	        Easy        Hard
Interview-friendly	YES	        Meh
9️⃣ Pandas Mental Mapping (IMPORTANT)
SQL CTE	Pandas
WITH base AS (...)	df_base = df[...]
Next CTE	df2 = df_base.assign(...)
Final SELECT	df_final

SQL CTEs = DataFrames chained together

🔑 One Sentence You Must Remember

CTEs let you build complex features step-by-step without repeating logic.

Say this in interviews. Period.

In [20]:
print("Sub topic : Joins optimization (when to use which type)")

Sub topic : Joins optimization (when to use which type)


📘 JOIN Optimization for Feature Engineering (Copy-Paste Theory)

JOINs combine features from multiple tables.
Optimizing JOINs means choosing the correct JOIN type to avoid unnecessary rows, NULLs, and slow queries.

1️⃣ INNER JOIN (Most Efficient)

Definition:
Returns rows that exist in both tables.

When to use:

When missing data is NOT allowed

When both sides must match

Best performance (smallest result set)

Feature Engineering Use:
✔ User with transactions
✔ Products with sales

2️⃣ LEFT JOIN (Most Common for ML)

Definition:
Returns all rows from left table and matching rows from right table.

When to use:

When you cannot lose rows

When right table provides optional features

Default choice for feature engineering

Feature Engineering Use:
✔ Users without transactions
✔ Customers without activity

3️⃣ RIGHT JOIN (Usually Avoided)

Definition:
Returns all rows from right table.

Why avoid:

Confusing

Equivalent to LEFT JOIN by swapping tables

4️⃣ FULL JOIN (Rare in ML)

Definition:
Returns all rows from both tables.

Why avoid:

Many NULLs

Hard to model

Often unnecessary

5️⃣ CROSS JOIN (Almost Never)

Definition:
Cartesian product.

Why avoid:

Explodes rows

Memory killer

🔑 JOIN Optimization Rules (IMPORTANT)

1️⃣ Always filter before JOIN
2️⃣ JOIN on indexed / primary keys
3️⃣ Prefer INNER JOIN when possible
4️⃣ LEFT JOIN for feature tables
5️⃣ Avoid FULL and CROSS JOIN unless required

In [21]:
users = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "country": ["PK", "PK", "US", "UK"]
})

users.to_sql("users", conn, index=False, if_exists="replace")


4

In [22]:
transactions = pd.DataFrame({
    "user_id": [1, 1, 2, 3],
    "amount": [100, 150, 200, 300]
})

transactions.to_sql("transactions", conn, index=False, if_exists="replace")


4

In [23]:
query = """
SELECT
    u.user_id,
    u.country,
    t.amount
FROM users u
INNER JOIN transactions t
ON u.user_id = t.user_id;
"""

pd.read_sql(query, conn)


,user_id,country,amount
0,1,PK,100
1,1,PK,150
2,2,PK,200
3,3,US,300


In [24]:
query = """
SELECT
    u.user_id,
    u.country,
    t.amount
FROM users u
LEFT JOIN transactions t
ON u.user_id = t.user_id;
"""

pd.read_sql(query, conn)


,user_id,country,amount
0,1,PK,100.0
1,1,PK,150.0
2,2,PK,200.0
3,3,US,300.0
4,4,UK,NaN


In [25]:
query = """
WITH txn_features AS (
    SELECT
        user_id,
        SUM(amount) AS total_spent
    FROM transactions
    GROUP BY user_id
)

SELECT
    u.user_id,
    u.country,
    t.total_spent
FROM users u
LEFT JOIN txn_features t
ON u.user_id = t.user_id;
"""

pd.read_sql(query, conn)


,user_id,country,total_spent
0,1,PK,250.0
1,2,PK,200.0
2,3,US,300.0
3,4,UK,NaN


5️⃣ Aggregation BEFORE JOIN (Optimization)

❌ BAD (join → then aggregate):

SELECT ...
FROM users
LEFT JOIN transactions;


✔ GOOD (aggregate → then join):

query = """
WITH txn_features AS (
    SELECT
        user_id,
        SUM(amount) AS total_spent
    FROM transactions
    GROUP BY user_id
)

SELECT
    u.user_id,
    u.country,
    t.total_spent
FROM users u
LEFT JOIN txn_features t
ON u.user_id = t.user_id;
"""

pd.read_sql(query, conn)


🔥 Faster
🔥 Cleaner
🔥 ML-ready

6️⃣ JOIN Type Decision Table (MEMORIZE)
Situation	JOIN
Mandatory match	INNER JOIN
Base table must not shrink	LEFT JOIN
Optional features	LEFT JOIN
Performance critical	INNER JOIN
Full data union	FULL JOIN (rare)
7️⃣ Interview-Ready Line (Say This)

“In feature engineering, the base entity should never shrink, so LEFT JOIN is preferred. INNER JOIN is used only when missing data is unacceptable.”